# Day 3 Hands-On Laboratory: Transverse Spectra, Feynman Scaling, and Radial Flow
===============================================================================

Welcome to the third laboratory of the AMPT course series. In this laboratory, we will explore the transverse dynamics of relativistic heavy-ion collisions, Feynman scaling in the longitudinal direction, and collective radial flow expansion.

We will map the theoretical concepts discussed in **Chapter 5, Sections 5.2.5.2, 5.2.6, 5.2.7, and 5.2.8** of Sahoo's textbook to real data-driven python programs utilizing AMPT datasets.

### Objectives:
1. Calculate accelerator luminosity and runtime statistics for collider configurations.
2. Implement and verify Feynman scaling ($x_F$) distributions for produced pions.
3. Parse AMPT datasets, extract identified particle species, and calculate their invariant yields.
4. Fit transverse mass ($m_T - m_0$) spectra to extract the mass-dependent effective temperature $T_{eff}$ and determine the thermal freeze-out temperature $T_{th}$ and collective radial flow velocity $\langle v_{flow} \rangle$.
5. Plot and analyze event-by-event average transverse momentum $\langle p_T \rangle$ vs. charged particle multiplicity density $dN_{ch}/d\eta$ (the Van Hove-like collectivity signature).

## Problem 1: Accelerator Luminosity and Runtime Statistics

From Sahoo Eq. 5.188, the luminosity $\mathcal{L}$ of a collider experiment with counter-propagating symmetric beams is:
$$\mathcal{L} = \frac{f n_b N_p^2}{4\pi\sigma_x\sigma_y}$$
where:
- $f$ is the bunch revolution frequency (in Hz)
- $n_b$ is the number of bunches per beam
- $N_p$ is the number of protons/nucleons per bunch
- $\sigma_x, \sigma_y$ are the horizontal and vertical beam profile widths at the interaction point (in cm)

### Your Tasks:
1. Implement `calculate_luminosity(f, n_b, N_p, sigma_x, sigma_y)` to return the luminosity (in $\mathrm{cm}^{-2}\mathrm{s}^{-1}$).
2. For LHC proton-proton collision settings:
   - $f = 11245$ Hz
   - $n_b = 2808$
   - $N_p = 1.15 \times 10^{11}$ protons/bunch
   - $\sigma_x = \sigma_y = 16\ \mu\mathrm{m} = 1.6 \times 10^{-3}$ cm
   Calculate and print the LHC luminosity.
3. Given the inelastic proton-proton cross-section $\sigma_{inel} = 80$ mb ($80 \times 10^{-27}\ \mathrm{cm}^2$), the event rate is $\mathrm{d}R/\mathrm{d}t = \mathcal{L}\sigma_{inel}$ (Sahoo Eq. 5.190). Calculate the number of events produced in a run of 15 hours.
4. Calculate how many hours are required to collect $N_{events} = 2.5 \times 10^9$ events. (Separately, calculate the number of events $N$ required to achieve a $2\%$ statistical error on the total count, where $N = 1/\text{error}^2$, and compute the hours needed to collect that smaller sample).

In [ ]:
import numpy as np

def calculate_luminosity(f, n_b, N_p, sigma_x, sigma_y):
    # TODO: Implement Sahoo Eq. 5.188
    pass

# 1. LHC parameters
f_lhc = 11245.0  # Hz
n_b_lhc = 2808
N_p_lhc = 1.15e11
sigma_lhc = 16e-4  # 16 micrometers in cm

# Calculate and print LHC luminosity
# TODO

# 2. Calculate event rate and total events in 15 hours
sigma_inel = 80e-3  # 80 mb = 80e-3 barns (convert to cm^2 using 1 barn = 1e-24 cm^2)
# TODO

# 3. Run time required for 2% statistical error
# TODO

## Problem 2: Feynman scaling variable $x_F$

Feynman introduced the dimensionless scaling variable $x_F$ (Sahoo Eq. 5.192):
$$x_F = \frac{p_z}{p_z(\mathrm{max})} \approx \frac{2p_z}{\sqrt{s}}$$
where the $2p_z/\sqrt{s}$ form is a high-energy approximation assuming negligible particle mass.

According to the Feynman scaling hypothesis, at very high energies, the longitudinal momentum distributions of produced secondary particles scale and become independent of the collision energy.

### Your Tasks:
1. Read the AMPT subset file for 39 GeV (`../Data/subsets/ampt_39_sub100.dat`).
2. Select charged pions ($\pi^\pm$, PID $\pm 211$) and compute $x_F = 2p_z / 39.0$ for each.
3. Plot the histogram of $x_F$ in the range $[-0.2, 0.2]$ (use 40 bins).
4. Highlight the mid-rapidity region ($|y| < 0.5$) on the same plot to show that the central rapidity plateau corresponds to the $x_F \approx 0$ region.

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')
from ampt_parser import iter_events
from kinematics import rapidity
import matplotlib.pyplot as plt

filepath = "../Data/subsets/ampt_39_sub100.dat"
sqrt_s = 39.0

all_xf = []
mid_rap_xf = []

# TODO: Loop over events, extract pions, calculate x_F and rapidity y
# Accumulate x_F values

# TODO: Plot histograms comparing all pions vs. mid-rapidity pions (|y| < 0.5)
fig, ax = plt.subplots(figsize=(8, 6))

ax.set_xlabel(r'Feynman Scaling Variable $x_F$', fontsize=14)
ax.set_ylabel('Raw counts', fontsize=14)
ax.set_title(r'Feynman $x_F$ Distribution for Pions at $\sqrt{s_{NN}} = 39$ GeV', fontsize=14)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()

## Problem 3: Parsing AMPT and Extracting Invariant Yields

The Lorentz-invariant yield is defined as (Sahoo §5.2.7):
$$f(p_T) = \frac{1}{2\pi p_T} \frac{\mathrm{d}^2N}{\mathrm{d}p_T \mathrm{d}y} = \frac{1}{2\pi m_T} \frac{\mathrm{d}^2N}{\mathrm{d}m_T \mathrm{d}y}$$
We can approximate the derivative by binning tracks in transverse momentum $p_T$ and normalizing:
$$\frac{1}{2\pi p_T} \frac{\mathrm{d}^2N}{\mathrm{d}p_T \mathrm{d}y} \approx \frac{1}{2\pi p_{T,\mathrm{bin}} \Delta p_T \Delta y N_{\mathrm{ev}}} N_{\mathrm{tracks}}$$

### Your Tasks:
1. Iterate over the 39 GeV dataset and extract $p_T$ for pions (PID $\pm 211$), kaons (PID $\pm 321$), and protons (PID $\pm 2212$) in $|y| < 0.5$.
2. Bin the particles in $p_T$ from 0 to 3.0 GeV (30 bins, width 0.1 GeV).
3. Calculate the invariant yield and its statistical error (assuming $\sigma_{N} = \sqrt{N_{\mathrm{tracks}}}$) for each species.
4. Plot the yields on a log-y scale. Observe the different shapes.

In [ ]:
from kinematics import transverse_momentum

species_info = {
    'pion':   {'pid': 211,  'mass': 0.139570, 'label': r'$\pi^{\pm}$',    'color': 'blue',   'marker': 'o'},
    'kaon':   {'pid': 321,  'mass': 0.493677, 'label': r'$K^{\pm}$',     'color': 'green',  'marker': 's'},
    'proton': {'pid': 2212, 'mass': 0.938272, 'label': r'$p/\bar{p}$',   'color': 'red',    'marker': '^'}
}

y_cut = 0.5
dy = 2 * y_cut
pt_bins = np.linspace(0.0, 3.0, 31)
pt_width = pt_bins[1] - pt_bins[0]
pt_centers = 0.5 * (pt_bins[:-1] + pt_bins[1:])

fig, ax = plt.subplots(figsize=(8, 6))

for name, info in species_info.items():
    all_pt = []
    event_count = 0
    
    # TODO: Loop over events, filter by species PID and rapidity y cut
    # Compute pT and accumulate
    # Compute invariant yield and error, plot on log-y
    pass

ax.set_yscale('log')
ax.set_xlabel(r'$p_T$ (GeV/$c$)', fontsize=14)
ax.set_ylabel(r'Invariant Yield $\frac{1}{2\pi p_T} \frac{\mathrm{d}^2N}{\mathrm{d}p_T\mathrm{d}y}$ (GeV/$c$)$^{-2}$', fontsize=14)
ax.set_xlim(0.0, 3.0)
ax.legend(frameon=True, fontsize=12)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()

## Problem 4: Boltzmann Fitting and Radial Flow Extraction
👉 **[INTERACTIVE SIMULATOR]:** Open the simulator **[Transverse Spectra & Radial Flow Analyzer](../animations/09_pt_mt_spectra_analyzer.html)** in your browser. Toggle between $p_T$ and $m_T - m_0$ plot variables, and sweep the $v_{flow}$ slider. Notice how the linear mass-dependent slopes are generated!


At low $p_T$ (the thermal regime), the transverse mass spectrum follows an exponential Boltzmann-type distribution (Sahoo Eq. 5.208 / 5.214):
$$\frac{1}{2\pi m_T} \frac{\mathrm{d}^2N}{\mathrm{d}m_T \mathrm{d}y} = C e^{-(m_T - m_0)/T_{eff}}$$
where $T_{eff}$ is the effective slope parameter (effective temperature). By taking the natural logarithm, we linearize the equation:
$$\ln\left( \text{Yield} \right) = \ln(C) - \frac{1}{T_{eff}} (m_T - m_0)$$

According to a simplified, non-relativistic 2D radial expansion model (Sahoo Eq. 5.222), the effective temperature rises approximately linearly with mass due to collective velocity superposition:
$$T_{eff} = T_{th} + \frac{1}{2} m_0 \langle v_{flow} \rangle^2$$
where $T_{th}$ is the thermal freeze-out temperature and $\langle v_{flow} \rangle$ is the average radial expansion velocity. Note that this is a simplified heuristic model; real experimental extractions use the full relativistic blast-wave formulation.
### Your Tasks:
1. Extract the $m_T - m_0$ spectra for pions, kaons, and protons in $|y| < 0.5$ using the 39 GeV dataset.
2. Perform a linear fit using `np.polyfit` to the logarithm of the invariant yield in the fit range $m_T - m_0 \in [0.0, 1.0]$ GeV.
3. Extract and print $T_{eff}$ (equal to $-1/\text{slope}$) for the three species.
4. Plot $T_{eff}$ vs. mass $m_0$ and perform a linear fit to extract $T_{th}$ (intercept) and $\langle v_{flow} \rangle = \sqrt{2 \times \text{slope}}$.

In [ ]:
from kinematics import transverse_mass

mt_bins = np.linspace(0.0, 1.5, 31)
mt_width = mt_bins[1] - mt_bins[0]
mt_centers = 0.5 * (mt_bins[:-1] + mt_bins[1:])

teff_results = {}

for name, info in species_info.items():
    all_mt_diff = []
    event_count = 0
    
    # TODO: Loop over events, extract mT - m0 and calculate invariant yield
    # Filter bins for mT - m0 < 1.0 GeV
    # Perform linearized fit: Y = ln(Yield), X = mT - m0
    # slope, intercept = np.polyfit(X_fit, Y_fit, 1)
    # teff = -1.0 / slope
    # teff_results[name] = teff
    pass

# Extracted slope parameter linear fit
masses = np.array([species_info[name]['mass'] for name in teff_results.keys()])
teffs = np.array([teff_results[name] for name in teff_results.keys()])

# TODO: Fit Teff vs mass: Y = teffs, X = masses
# slope, intercept = np.polyfit(masses, teffs, 1)
# T_th = intercept
# v_flow = np.sqrt(2 * slope)

# TODO: Plot Teff vs mass showing data points and the fit line

## Problem 5: Event-by-event Multiplicity Correlation: Van Hove-like Signature
👉 **[INTERACTIVE SIMULATOR]:** Open the simulator **[Van Hove Phase Transition Signature](../animations/10_van_hove_phase_transition.html)** in your browser. Sweep the multiplicity density and watch the physical fireball transition through the mixed phase plateau. Notice how the temperature profile remains flat due to latent heat!


Plotting the event-by-event average transverse momentum $\langle p_T \rangle$ vs. charged particle multiplicity density $dN_{ch}/d\eta$ allows us to probe the collective expansion of the produced hot QCD matter. A plateau in this profile qualitatively resembles the Van Hove signature of mixed-phase coexistence during a first-order phase transition. Note that standard AMPT does not incorporate a thermodynamic first-order phase transition; in the model, this plateau arises from parton cascade dynamics, hadronic scattering, and saturation effects.### Your Tasks:
1. Calculate event-by-event $dN_{ch}/d\eta$ (in $|\eta| < 0.5$) and $\langle p_T \rangle$ (for charged particles in $|\eta| < 0.5$) for the 7.7 GeV and 39 GeV datasets.
2. Bin events by their $dN_{ch}/d\eta$ values, and compute the average $\langle p_T \rangle$ in each multiplicity bin.
3. Plot the profile curves for both energies on a single canvas, add proper legends, labels, and discuss the presence of a plateau.

In [ ]:
from ampt_parser import is_charged
from kinematics import pseudorapidity

# TODO: Parse events for both files
# Calculate event-by-event Nch and mean pT
# Bin events by multiplicity, compute profile averages
# Plot profile curves on a single canvas with annotations

fig, ax = plt.subplots(figsize=(8, 6))
# ax.errorbar(...)
ax.set_xlabel(r'Multiplicity Density $\mathrm{d}N_{\mathrm{ch}}/\mathrm{d}\eta$', fontsize=14)
ax.set_ylabel(r'Average Transverse Momentum $\langle p_T \rangle$ (GeV/$c$)', fontsize=14)
ax.set_title(r'Van Hove-like Signature: $\langle p_T \rangle$ vs. Multiplicity Density', fontsize=14)
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()